# Tutorial 3: Spec Contracts and Debugging

Estimated time: 25-35 minutes

## Prerequisites
None.

## Learning aims
- Read pydantic validation errors and fix typed-spec mistakes
- Use `mm validate` and `mm doctor` to triage problems quickly


In [ ]:
# Cross-platform setup — works on Windows / macOS / Linux.
# Self-contained: walks up to find the repo root, adds src/ to sys.path,
# then imports the official bootstrap helper for chdir + later run_cli use.
import os, sys
from pathlib import Path

_here = Path.cwd().resolve()
for _candidate in [_here, *_here.parents]:
    _marker = _candidate / 'pyproject.toml'
    if _marker.is_file() and 'name = "metamodeler"' in _marker.read_text():
        ROOT = _candidate
        break
else:
    raise FileNotFoundError('Could not locate metamodeler repo root from ' + str(_here))

_src = str((ROOT / 'src').resolve())
if _src not in sys.path:
    sys.path.insert(0, _src)
if Path.cwd().resolve() != ROOT.resolve():
    os.chdir(ROOT)

from metamodeler.tutorial import bootstrap, run_cli  # noqa: E402
ROOT = bootstrap()
print('Repo root:', ROOT)


## Step 1: Trigger a deliberate validation failure

In [ ]:
import json

bad_spec = ROOT / 'tmp/tutorial_bad_spec.json'
bad_spec.parent.mkdir(parents=True, exist_ok=True)
bad_spec.write_text(json.dumps({
    'schema_version': '1.0',
    'name': 'oops',
    # missing required fields on purpose
}))
result = run_cli('validate', str(bad_spec.relative_to(ROOT)), capture=True)
print('exit code:', result.returncode)
print(result.stdout)
print(result.stderr)


## Step 2: Read the error message and fix the spec

In [ ]:
fixed_spec_payload = json.loads(
    (ROOT / 'tutorials/specs/model.toy.grid.json').read_text()
)
fixed_path = ROOT / 'tmp/tutorial_fixed_spec.json'
fixed_path.write_text(json.dumps(fixed_spec_payload, indent=2))
run_cli('validate', str(fixed_path.relative_to(ROOT)))


## Step 3: Compare validation outputs

In [ ]:
from pathlib import Path

for label, p in [('BAD', bad_spec), ('FIXED', fixed_path)]:
    r = run_cli('validate', str(p.relative_to(ROOT)), capture=True)
    print(f'--- {label} (exit={r.returncode}) ---')
    print(r.stdout or r.stderr)


## Common debugging tips
- Always start with `mm doctor` to confirm the env is sane.
- Pydantic errors point at the field path; read the leaf message first.
- If JSON parsing itself fails, the line/column is in the error text.
